In [3]:
# !pip install rank-bm25
# !pip install nltk

In [4]:
# ==========================================================
# Notebook 03: BM25 Lexical Search Engine
# ==========================================================

import re

import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi

In [5]:
DATA_PATH = "data/processed/enterprise_corpus_acl.csv"

enterprise_df = pd.read_csv(DATA_PATH)

enterprise_df.head()

,source,title,content,author,doc_id,department,created_date,tags,acl_roles,security_level,owner_team
0,Slack,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...,Alice,DOC0001,Engineering,2026-01-11,"['enterprise-search', 'semantic-search']","['Engineering', 'Public']",Internal,Engineering
1,Slack,Model Training Update,The semantic search model has been retrained u...,Bob,DOC0002,Finance,2026-01-12,"['phoenix', 'kubernetes']","['Finance', 'Public']",Restricted,Finance
2,Confluence,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...,Charlie,DOC0003,HR,2026-01-13,"['semantic-search', 'enterprise-search']","['HR', 'Public']",Restricted,HR
3,Confluence,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...,Diana,DOC0004,Engineering,2026-01-14,"['fastapi', 'acl']","['Engineering', 'Public']",Internal,Engineering
4,Jira,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...,Eric,DOC0005,Engineering,2026-01-15,"['enterprise-search', 'kubernetes']","['Engineering', 'Public']",Internal,Engineering


In [6]:
enterprise_df[["doc_id", "title", "content"]].head()

,doc_id,title,content
0,DOC0001,Phoenix Deployment Discussion,The Phoenix-2026 deployment is scheduled for n...
1,DOC0002,Model Training Update,The semantic search model has been retrained u...
2,DOC0003,Phoenix Architecture Wiki,Project Phoenix-2026 follows a microservice ar...
3,DOC0004,Enterprise Search Design,Hybrid search combines BM25 keyword retrieval ...
4,DOC0005,PHX-245 Database Migration,Complete PostgreSQL migration before Phoenix p...


In [7]:
def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9\s\-]", " ", text)

    tokens = text.split()

    return tokens

In [8]:
sample = "Phoenix-2026 deployment is scheduled!"

print(preprocess_text(sample))

['phoenix-2026', 'deployment', 'is', 'scheduled']


In [9]:
enterprise_df["search_text"] = enterprise_df["title"] + " " + enterprise_df["content"]

In [10]:
enterprise_df[["doc_id", "search_text"]].head()

,doc_id,search_text
0,DOC0001,Phoenix Deployment Discussion The Phoenix-2026...
1,DOC0002,Model Training Update The semantic search mode...
2,DOC0003,Phoenix Architecture Wiki Project Phoenix-2026...
3,DOC0004,Enterprise Search Design Hybrid search combine...
4,DOC0005,PHX-245 Database Migration Complete PostgreSQL...


In [11]:
tokenized_corpus = [
    preprocess_text(document) for document in enterprise_df["search_text"]
]

In [12]:
tokenized_corpus[0]

['phoenix',
 'deployment',
 'discussion',
 'the',
 'phoenix-2026',
 'deployment',
 'is',
 'scheduled',
 'for',
 'next',
 'friday',
 'database',
 'migration',
 'will',
 'happen',
 'during',
 'the',
 'maintenance',
 'window']

In [13]:
bm25 = BM25Okapi(tokenized_corpus)

In [14]:
def bm25_search(query, bm25_model, dataframe, top_k=5):

    query_tokens = preprocess_text(query)

    scores = bm25_model.get_scores(query_tokens)

    results = dataframe.copy()

    results["bm25_score"] = scores

    results = (
        results.sort_values(by="bm25_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    return results

In [15]:
query = "Phoenix-2026 deployment"

results = bm25_search(query, bm25, enterprise_df, top_k=3)

results[["doc_id", "title", "bm25_score"]]

,doc_id,title,bm25_score
0,DOC0001,Phoenix Deployment Discussion,2.137921
1,DOC0003,Phoenix Architecture Wiki,0.597749
2,DOC0002,Model Training Update,0.000000


In [16]:
query = "PHX-245"

ticket_results = bm25_search(query, bm25, enterprise_df, top_k=3)

ticket_results[["doc_id", "title", "bm25_score"]]

,doc_id,title,bm25_score
0,DOC0005,PHX-245 Database Migration,1.470886
1,DOC0001,Phoenix Deployment Discussion,0.000000
2,DOC0002,Model Training Update,0.000000


In [17]:
queries = ["Phoenix deployment", "semantic search", "ACL filtering", "PHX-245"]

for query in queries:

    print("=" * 70)

    print(f"QUERY: {query}")

    output = bm25_search(query, bm25, enterprise_df, top_k=2)

    print(output[["title", "bm25_score"]])

    print()

QUERY: Phoenix deployment
                           title  bm25_score
0  Phoenix Deployment Discussion      1.6412
1          Model Training Update      0.0000

QUERY: semantic search
                      title  bm25_score
0  Enterprise Search Design    0.597749
1     Model Training Update    0.559797

QUERY: ACL filtering
                           title  bm25_score
0        SEC-104 ACL Enhancement    2.834799
1  Phoenix Deployment Discussion    0.000000

QUERY: PHX-245
                           title  bm25_score
0     PHX-245 Database Migration    1.470886
1  Phoenix Deployment Discussion    0.000000



In [18]:
query = "Phoenix"

query_tokens = preprocess_text(query)

scores = bm25.get_scores(query_tokens)

score_df = pd.DataFrame(
    {
        "doc_id": enterprise_df["doc_id"],
        "title": enterprise_df["title"],
        "score": scores,
    }
)

score_df.sort_values(by="score", ascending=False)

,doc_id,title,score
0,DOC0001,Phoenix Deployment Discussion,0.0
1,DOC0002,Model Training Update,0.0
2,DOC0003,Phoenix Architecture Wiki,0.0
3,DOC0004,Enterprise Search Design,0.0
4,DOC0005,PHX-245 Database Migration,0.0
5,DOC0006,SEC-104 ACL Enhancement,0.0


In [19]:
import pickle
import os

OUTPUT_PATH = "artifacts/"

os.makedirs(OUTPUT_PATH, exist_ok=True)

with open(OUTPUT_PATH + "bm25_model.pkl", "wb") as file:

    pickle.dump(bm25, file)

print("BM25 model saved!")

BM25 model saved!


In [20]:
with open("artifacts/bm25_model.pkl", "rb") as file:

    loaded_bm25 = pickle.load(file)

print("BM25 model loaded!")

BM25 model loaded!
